## Side-by-side timeseries animations for Land Cover 2.0 and Geomedian

This notebook uses the [Land Cover 3.0](https://knowledge.dea.ga.gov.au/data/product/dea-land-cover-landsat/) and [Geometric Median and Median Absolute Deviation (Landsat)](https://knowledge.dea.ga.gov.au/data/product/dea-geometric-median-and-median-absolute-deviation-landsat/) products to create animations showing areas of interest throughout time in yearly timesteps.

This notebook connects to the development sandbox database currently; once Land Cover 3.0 is publicly accessible this can be updated and access to the database replaced with use of the `datacube` python package for accessing the datasets.

The notebook has a cell for the user to input parameters. These will then be passed to functions that will connect to the database, generate individual animations, and then generate paired animations. These will be saved out to the directory specified in the parameter cell.


In [21]:
%matplotlib inline

import os
import sys
import pandas as pd
import numpy as np
import datacube
import geopandas as gpd
import rasterio.features
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from datacube.utils.masking import make_mask
from matplotlib import colors as mcolours
from IPython.display import Image
from IPython.core.display import Video
from shapely.geometry import shape

from PIL import ImageSequence
from PIL import Image as pimg

sys.path.insert(1, "../Tools/")
from dea_tools.plotting import xr_animation
from dea_tools.landcover import lc_animation, lc_colourmap

The cell below is only required if trying to access data that is only in the development database. If you do not need to do that, you can comment out the cell and proceed using `datacube`

In [6]:

dc = datacube.Datacube(app="landcover_geomad_aminations", env='dev')

In [7]:
# # If you do not need to access the dev sandbox use the following instead:

# dc = datacube.Datacube(app="landcover_geomad_aminations")

Place parameters in the cell below. Input parameters that change for each target location are placed in lists.

- `output_dir`: the directory where the individual animations will be saved
- `text_size`: size of the 'year' text
- `dpi`
- `lc_product`: the land cover product
- `gmad_product`: the geoMAD product used. In this case, the Landsat 7 GeoMAD.
- `buffers`: list of buffers used with central lat/lon coordinates to generate region of interest
- `intervals`: Control the speed of the animations by changing the itnerval between each frame.
- `lat_list`: list of the central latitude for each area of interest
- `lon_list`: list of the central longitude for each area of interest
- `time_list`: a list of tuples, each containing the start and end year for the desired timeframe
- `roi_name_list`: list containing a string name that will be appended to the output filenames

In [8]:
output_dir = 'output_gifs'

text_size = 35
dpi = 150
lc_product = 'ga_ls_landcover_class_cyear_3'
gmad_product = 'ga_ls7e_gm_cyear_3'
aspect_ratio = 0.9

#list parameters. Each index in the list represents a different area for an animation to be created at.
buffers = [0.07]#, 0.1, 0.03, 0.15, 0.1]
intervals = [800]#, 400, 400, 400, 400]
lat_list = [-37.56127]#, -35.1775, -38.85991, -24.79203, -16.97106]
lon_list = [145.32133]#, 149.09052, 146.17321, 130.99059, 145.78298]
time_list = [('2005', '2015')]#, ('2000', '2022'), ('2000', '2022'), ('2000', '2022'), ('2000', '2022')]

roi_name_list = ['mt_beggary_bushfire']#, 'canberra_urban_expansion', 'shallow_inlet', 'lake_amadeus', 'cairns_trinity_inlet']

In [9]:
def odc_connect(lat, lon, buffer, time):
    """
    Connects to the Open Data Cube (ODC) and loads datasets for land cover and geomad products within a specified geographic and temporal range.

    Parameters:
    lat (float): Latitude of the center point
    lon (float): Longitude of the center point
    buffer (float): Buffer distance around the center point to define the area of interest
    time (str): Time range for the data query in the format ('YYYY, YYYY')

    Returns:
    tuple: A tuple containing:
        - ds_lc (xarray.Dataset): Dataset containing land cover data
        - ds_gmad (xarray.Dataset): Dataset containing geomad data
    """
    lat_range = (lat - buffer, lat + buffer)
    lon_range = (lon - buffer, lon + buffer)
    
    query ={
        'x': lon_range,
        'y': lat_range,
        'time': time
    }
    ds_lc = dc.load(product=lc_product,
                measurements=['level3', 'level4'],
                **query)
    ds_gmad = dc.load(product=gmad_product,
                  measurements=['nbart_red', 'nbart_green', 'nbart_blue'],
                  **query)
    
    return ds_lc, ds_gmad

In [10]:
def generate_single_animations(ds, gmad_ds, roi_name, interval, lc_class=3):
    """
    Generates GIF animations for land cover levels 3 and 4, and geomad data.

    Parameters:
    ds (xarray.Dataset): Dataset containing land cover data.
    gmad_ds (xarray.Dataset): Dataset containing geomad data.
    roi_name (str): Name of the region of interest.
    interval (int): Interval between frames in the animation.
    lc_class (int): level 3 or level 4 land cover collection 3

    Returns:
    tuple: A tuple containing:
        - file_name_gmad (str): File path of the generated geomad animation GIF.
        - file_name_landcover (str): File path of the generated level 3 land cover animation GIF.
    """

    final_animations_dir = os.path.join(output_dir, 'final_animations')
    file_name_gmad = f'{output_dir}/{roi_name}_timeseries_geomad.gif'

    if not os.path.exists(final_animations_dir):
        os.makedirs(final_animations_dir)

    #generate landcover animation, using the lc_class to determine if its level3 or level4
    if lc_class == 3:
        file_name_landcover = f'{output_dir}/{roi_name}_timeseries_lc_lvl3.gif'
        lc_animation(ds.level3,
                    file_name=f'{output_dir}/{roi_name}_timeseries_lc_lvl3',
                    colour_bar=False,
                    label_ax=False,
                    animation_interval=interval,
                    width_pixels=7,
                    font_size=text_size,
                    dpi=dpi)
        
    elif lc_class == 4: 
        file_name_landcover = f'{output_dir}/{roi_name}_timeseries_lc_lvl4.gif'
        lc_animation(ds.level4,
                    file_name=f'{output_dir}/{roi_name}_timeseries_lc_lvl4',
                    colour_bar=False,
                    label_ax=False,
                    animation_interval=interval,
                    width_pixels=7,
                    font_size=text_size,
                    dpi=dpi)

    
    #generate geomad animations
    xr_animation(ds=gmad_ds,
                bands=['nbart_red', 'nbart_green','nbart_blue'],
                output_path=f'{output_dir}/{roi_name}_timeseries_geomad.gif',
                interval=interval,
                width_pixels=350,
                show_colorbar=False,
                show_date = False,
                percentile_stretch=(0.01, 0.99),
                annotation_kwargs= {'fontsize': 25})
    plt.close()

    return file_name_gmad, file_name_landcover


In [11]:
def get_gif_dimensions(file_path):
    with pimg.open(file_path) as img:
        width, height = img.size
        return width, height

In [12]:
def crop_to_aspect_ratio(frame, aspect_ratio):
    """
    Crop pixels from the left and bottom of the gif frames to ensure all animations have the same aspect ratios.
    """
    width, height = frame.size
    new_height = int(width / aspect_ratio)
    new_width = width

    # Calculate the amount to crop from the left
    left_crop = (width - new_width) // 2

    # Crop the frame
    return frame.crop((left_crop, 0, width, new_height))

In [13]:
def timeseries_animation(file_name_gmad, file_name_landcover, aspect_ratio=None, crop=False):
    """
    Creates a side-by-side GIF animation combining geomad and land cover animations, ensuring they are synchronized.

    Parameters:
    file_name_gmad (str): File path of the geomad animation GIF.
    file_name_landcover (str): File path of the land cover animation GIF.
    aspect_ratio (float): The desired aspect ratio to crop to (width/height).

    Returns:
    str: File path of the combined animation GIF.
    """

    directory, filename = os.path.split(file_name_landcover)
    name, ext = os.path.splitext(filename)
    new_filename = f"{name}_gmad{ext}"
    output_filepath = os.path.join(directory, 'final_animations', new_filename)

    gif_lc = pimg.open(file_name_landcover)
    gif_gmad = pimg.open(file_name_gmad)

    if crop is True:
        frames1 = [crop_to_aspect_ratio(frame, aspect_ratio) for frame in ImageSequence.Iterator(gif_gmad)]
        frames2 = [crop_to_aspect_ratio(frame, aspect_ratio) for frame in ImageSequence.Iterator(gif_lc)]

    else:
        frames1 = [frame.copy() for frame in ImageSequence.Iterator(gif_gmad)]
        frames2 = [frame.copy() for frame in ImageSequence.Iterator(gif_lc)]
        

    # Get dimensions of cropped gifs
    frame_width, frame_height = frames1[0].size

    # Set the figure size dynamically based on the size of the gifs
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(frame_width * 2 / dpi, frame_height / dpi))

    # Function to make sure gif animations are synced
    def frame_update(frame):
        ax1.clear()
        ax2.clear()
        ax1.imshow(frames1[frame % len(frames1)])
        ax2.imshow(frames2[frame % len(frames2)])
        ax1.axis('off')
        ax2.axis('off')

    # Adjust layout to minimize white space
    plt.subplots_adjust(wspace=0, hspace=0, left=0, right=1, top=1, bottom=0)

    # Animation object
    animation_object = animation.FuncAnimation(fig, 
                                               frame_update, 
                                               frames=len(frames1),  
                                               interval=interval)  # Adjust interval as needed

    animation_object.save(output_filepath, writer="Pillow")
    plt.close()

    return output_filepath

In [14]:
for lat, lon, buffer, time, interval, roi in zip(lat_list, lon_list, buffers, time_list, intervals, roi_name_list):
    lc_ds, gm_ds = odc_connect(lat, lon, buffer, time)
    fname_gmad, fname_3 = generate_single_animations(lc_ds, gm_ds, roi, interval, lc_class=3)
    fname_gmad, fname_4 = generate_single_animations(lc_ds, gm_ds, roi, interval, lc_class=4)
    timeseries_animation(fname_gmad, fname_3, aspect_ratio, crop=True) #level 3 animation
    timeseries_animation(fname_gmad, fname_4, aspect_ratio, crop=True) #level 4 animation

Exporting animation to output_gifs/mt_beggary_bushfire_timeseries_geomad.gif


  0%|          | 0/11 (0.0 seconds remaining at ? frames/s)

Exporting animation to output_gifs/mt_beggary_bushfire_timeseries_geomad.gif


  0%|          | 0/11 (0.0 seconds remaining at ? frames/s)

MovieWriter Pillow unavailable; using Pillow instead.
MovieWriter Pillow unavailable; using Pillow instead.


In [36]:
import matplotlib.patheffects as PathEffects
from dea_tools.spatial import add_geobox
from shapely.geometry import box
from skimage.exposure import rescale_intensity
from matplotlib.animation import FuncAnimation
from pathlib import Path

def xr_animation_modified(ds,
                 bands=None,
                 output_path='animation.mp4',
                 width_pixels=500,
                 interval=100,
                 percentile_stretch=(0.02, 0.98),
                 image_proc_funcs=None,
                 show_gdf=None,
                 show_date='%d %b %Y',
                 show_text=None,
                 show_colorbar=True,
                 gdf_kwargs={},
                 annotation_kwargs={},
                 imshow_kwargs={},
                 colorbar_kwargs={},
                 limit=None,
                 list_extra_labels=[]):
    """
    Takes an `xarray` timeseries and animates the data as either a 
    three-band (e.g. true or false colour) or single-band animation, 
    allowing changes in the landscape to be compared across time.
    
    Animations can be customised to include text and date annotations 
    or use specific combinations of input bands. Vector data can be 
    overlaid and animated on top of imagery, and custom image 
    processing functions can be applied to each frame.
    
    Supports .mp4 (ideal for Twitter/social media) and .gif (ideal 
    for all purposes, but can have large file sizes) format files. 
    
    Last modified: April 2023
    
    Parameters
    ----------  
    ds : xarray.Dataset
        An xarray dataset with multiple time steps (i.e. multiple 
        observations along the `time` dimension).        
    bands : list of strings
        An list of either one or three band names to be plotted, 
        all of which must exist in `ds`. 
    output_path : str, optional
        A string giving the output location and filename of the 
        resulting animation. File extensions of '.mp4' and '.gif' are 
        accepted. Defaults to 'animation.mp4'.
    width_pixels : int, optional
        An integer defining the output width in pixels for the 
        resulting animation. The height of the animation is set 
        automatically based on the dimensions/ratio of the input 
        xarray dataset. Defaults to 500 pixels wide.        
    interval : int, optional
        An integer defining the milliseconds between each animation 
        frame used to control the speed of the output animation. Higher
        values result in a slower animation. Defaults to 100 
        milliseconds between each frame.         
    percentile_stretch : tuple of floats, optional
        An optional tuple of two floats that can be used to clip one or
        three-band arrays by percentiles to produce a more vibrant, 
        visually attractive image that is not affected by outliers/
        extreme values. The default is `(0.02, 0.98)` which is 
        equivalent to xarray's `robust=True`. This parameter is ignored
        completely if `vmin` and `vmax` are provided as kwargs to
        `imshow_kwargs`.
    image_proc_funcs : list of funcs, optional
        An optional list containing functions that will be applied to 
        each animation frame (timestep) prior to animating. This can 
        include image processing functions such as increasing contrast, 
        unsharp masking, saturation etc. The function should take AND 
        return a `numpy.ndarray` with shape [y, x, bands]. If your 
        function has parameters, you can pass in custom values using 
        a lambda function:
        `image_proc_funcs=[lambda x: custom_func(x, param1=10)]`.
    show_gdf: geopandas.GeoDataFrame, optional
        Vector data (e.g. ESRI shapefiles or GeoJSON) can be optionally
        plotted over the top of imagery by supplying a 
        `geopandas.GeoDataFrame` object. To customise colours used to
        plot the vector features, create a new column in the
        GeoDataFrame called 'colors' specifying the colour used to plot 
        each feature: e.g. `gdf['colors'] = 'red'`.
        To plot vector features at specific moments in time during the
        animation, create new 'start_time' and/or 'end_time' columns in
        the GeoDataFrame that define the time range used to plot each 
        feature. Dates can be provided in any string format that can be 
        converted using the `pandas.to_datetime()`. e.g.
         `gdf['end_time'] = ['2001', '2005-01', '2009-01-01']`    
    show_date : string or bool, optional
        An optional string or bool that defines how (or if) to plot 
        date annotations for each animation frame. Defaults to 
        '%d %b %Y'; can be customised to any format understood by 
        strftime (https://strftime.org/). Set to False to remove date 
        annotations completely.       
    show_text : str or list of strings, optional
        An optional string or list of strings with a length equal to 
        the number of timesteps in `ds`. This can be used to display a 
        static text annotation (using a string), or a dynamic title 
        (using a list) that displays different text for each timestep. 
        By default, no text annotation will be plotted.        
    show_colorbar : bool, optional
        An optional boolean indicating whether to include a colourbar 
        for single-band animations. Defaults to True.
    gdf_kwargs : dict, optional
        An optional dictionary of keyword arguments to customise the 
        appearance of a `geopandas.GeoDataFrame` supplied to 
        `show_gdf`. Keyword arguments are passed to `GeoSeries.plot` 
        (see http://geopandas.org/reference.html#geopandas.GeoSeries.plot). 
        For example: `gdf_kwargs = {'linewidth': 2}`. 
    annotation_kwargs : dict, optional
        An optional dict of keyword arguments for controlling the 
        appearance of  text annotations. Keyword arguments are passed 
        to `matplotlib`'s `plt.annotate` 
        (see https://matplotlib.org/api/_as_gen/matplotlib.pyplot.annotate.html 
        for options). For example, `annotation_kwargs={'fontsize':20, 
        'color':'red', 'family':'serif'}.  
    imshow_kwargs : dict, optional
        An optional dict of keyword arguments for controlling the 
        appearance of arrays passed to `matplotlib`'s `plt.imshow` 
        (see https://matplotlib.org/api/_as_gen/matplotlib.pyplot.imshow.html 
        for options). For example, a green colour scheme and custom
        stretch could be specified using: 
        `onebandplot_kwargs={'cmap':'Greens`, 'vmin':0.2, 'vmax':0.9}`.
        (some parameters like 'cmap' will only have an effect for 
        single-band animations, not three-band RGB animations).
    colorbar_kwargs : dict, optional
        An optional dict of keyword arguments used to control the 
        appearance of the colourbar. Keyword arguments are passed to
        `matplotlib.pyplot.tick_params` 
        (see https://matplotlib.org/api/_as_gen/matplotlib.pyplot.tick_params.html
        for options). This can be used to customise the colourbar 
        ticks, e.g. changing tick label colour depending on the 
        background of the animation: 
        `colorbar_kwargs={'colors': 'black'}`.
    limit: int, optional
        An optional integer specifying how many animation frames to 
        render (e.g. `limit=50` will render the first 50 frames). This
        can be useful for quickly testing animations without rendering 
        the entire time-series.  
    list_extra_labels: list, optional
        labels to add to animation, number of elements needs to be same of time steps in ds
            
    """

    def _start_end_times(gdf, ds):
        """
        Converts 'start_time' and 'end_time' columns in a 
        `geopandas.GeoDataFrame` to datetime objects to allow vector
        features to be plotted at specific moments in time during an
        animation, and sets default values based on the first
        and last time in `ds` if this information is missing from the
        dataset.
        """

        # Make copy of gdf so we do not modify original data
        gdf = gdf.copy()

        # Get min and max times from input dataset
        minmax_times = pd.to_datetime(ds.time.isel(time=[0, -1]).values)

        # Update both `start_time` and `end_time` columns
        for time_col, time_val in zip(['start_time', 'end_time'], minmax_times):

            # Add time_col if it does not exist
            if time_col not in gdf:
                gdf[time_col] = np.nan

            # Convert values to datetimes and fill gaps with relevant time value
            gdf[time_col] = pd.to_datetime(gdf[time_col], errors='ignore')
            gdf[time_col] = gdf[time_col].fillna(time_val)

        return gdf

    def _add_colorbar(fig, ax, vmin, vmax, imshow_defaults, colorbar_defaults):
        """
        Adds a new colorbar axis to the animation with custom minimum 
        and maximum values and styling.
        """

        # Create new axis object for colorbar
        cax = fig.add_axes([0.02, 0.02, 0.96, 0.03])

        # Initialise color bar using plot min and max values
        img = ax.imshow(np.array([[vmin, vmax]]), **imshow_defaults)
        fig.colorbar(img,
                     cax=cax,
                     orientation='horizontal',
                     ticks=np.linspace(vmin, vmax, 2))

        # Fine-tune appearance of colorbar
        cax.xaxis.set_ticks_position('top')
        cax.tick_params(axis='x', **colorbar_defaults)
        cax.get_xticklabels()[0].set_horizontalalignment('left')
        cax.get_xticklabels()[-1].set_horizontalalignment('right')

    def _frame_annotation(times, show_date, show_text):
        """
        Creates a custom annotation for the top-right of the animation
        by converting a `xarray.DataArray` of times into strings, and
        combining this with a custom text annotation. Handles cases 
        where `show_date=False/None`, `show_text=False/None`, or where
        `show_text` is a list of strings.
        """

        # Test if show_text is supplied as a list
        is_sequence = isinstance(show_text, (list, tuple, np.ndarray))

        # Raise exception if it is shorter than number of dates
        if is_sequence and (len(show_text) == 1):
            show_text, is_sequence = show_text[0], False
        elif is_sequence and (len(show_text) < len(times)):
            raise ValueError(f'Annotations supplied via `show_text` must have '
                             f'either a length of 1, or a length >= the number '
                             f'of timesteps in `ds` (n={len(times)})')

        times_list = (times.dt.strftime(show_date).values
                      if show_date else [None] * len(times))
        text_list = show_text if is_sequence else [show_text] * len(times)
        annotation_list = [
            '\n'.join([str(i)
                       for i in (a, b)
                       if i])
            for a, b in zip(times_list, text_list)
        ]

        return annotation_list

    def _update_frames(i, ax, extent, annotation_text, gdf, gdf_defaults,
                       annotation_defaults, imshow_defaults):
        """
        Animation called by `matplotlib.animation.FuncAnimation` to 
        animate each frame in the animation. Plots array and any text
        annotations, as well as a temporal subset of `gdf` data based
        on the times specified in 'start_time' and 'end_time' columns.
        """

        # Clear previous frame to optimise render speed and plot imagery
        ax.clear()
        ax.imshow(array[i, ...].clip(0.0, 1.0),
                  extent=extent,
                  vmin=0.0,
                  vmax=1.0,
                  **imshow_defaults)

        # Add annotation text
        ax.annotate(annotation_text[i], **annotation_defaults)

        # Add geodataframe annotation
        if show_gdf is not None:

            # Obtain start and end times to filter geodataframe features
            time_i = ds.time.isel(time=i).values

            # Subset geodataframe using start and end dates
            gdf_subset = show_gdf.loc[(show_gdf.start_time <= time_i) &
                                      (show_gdf.end_time >= time_i)]

            if len(gdf_subset.index) > 0:

                # Set color to geodataframe field if supplied
                if ('color' in gdf_subset) and ('color' not in gdf_kwargs):
                    gdf_defaults.update({'color': gdf_subset['color'].tolist()})

                gdf_subset.plot(ax=ax, **gdf_defaults)

        # Remove axes to show imagery only
        ax.axis('off')


    # Add GeoBox and odc.* accessor to array using `odc-geo`
    try:
        ds = add_geobox(ds)
    except ValueError:
        raise ValueError("Unable to determine `ds`'s coordinate "
                         "reference system (CRS). Please assign a CRS "
                         "to the array before passing it to this "
                         "function, e.g.: "
                         "`ds.odc.assign_crs(crs='EPSG:3577')`")
    
    # Test if bands have been supplied, or convert to list to allow
    # iteration if a single band is provided as a string
    if bands is None:
        raise ValueError(f'Please use the `bands` parameter to supply '
                         f'a list of one or three bands that exist as '
                         f'variables in `ds`, e.g. {list(ds.data_vars)}')
    elif isinstance(bands, str):
        bands = [bands]

    # Test if bands exist in dataset
    missing_bands = [b for b in bands if b not in ds.data_vars]
    if missing_bands:
        raise ValueError(f'Band(s) {missing_bands} do not exist as '
                         f'variables in `ds` {list(ds.data_vars)}')

    # Test if time dimension exists in dataset
    if 'time' not in ds.dims:
        raise ValueError(f"`ds` does not contain a 'time' dimension "
                         f"required for generating an animation")

    # Set default parameters
    outline = [PathEffects.withStroke(linewidth=2.5, foreground='black')]
    annotation_defaults = {
        'xy': (1, 1),
        'xycoords': 'axes fraction',
        'xytext': (-5, -5),
        'textcoords': 'offset points',
        'horizontalalignment': 'right',
        'verticalalignment': 'top',
        'fontsize': 20,
        'color': 'white',
        'path_effects': outline
    }
    imshow_defaults = {'cmap': 'magma', 'interpolation': 'nearest'}
    colorbar_defaults = {'colors': 'white', 'labelsize': 12, 'length': 0}
    gdf_defaults = {'linewidth': 1.5}

    # Update defaults with kwargs
    annotation_defaults.update(annotation_kwargs)
    imshow_defaults.update(imshow_kwargs)
    colorbar_defaults.update(colorbar_kwargs)
    gdf_defaults.update(gdf_kwargs)

    # Get info on dataset dimensions
    height, width = ds.odc.geobox.shape
    scale = width_pixels / width
    left, bottom, right, top = ds.odc.geobox.extent.boundingbox

    # Prepare annotations
    annotation_list = _frame_annotation(ds.time, show_date, show_text)

    if len(list_extra_labels)>0: # if a list of extra labels is provided
        annotation_list = [f'{a}\n{b}' for a,b in zip(annotation_list, list_extra_labels)]

    # Prepare geodataframe
    if show_gdf is not None:
        show_gdf = show_gdf.to_crs(ds.odc.geobox.crs)
        show_gdf = gpd.clip(show_gdf, mask=box(
            left, bottom, right, top)).reindex(show_gdf.index).dropna(how='all')
        show_gdf = _start_end_times(show_gdf, ds)

    # Convert data to 4D numpy array of shape [time, y, x, bands]
    ds = ds[bands].to_array().transpose(..., 'variable')[0:limit, ...]
    array = ds.astype(np.float32).values

    # Optionally apply image processing along axis 0 (e.g. to each timestep)
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} ({remaining_s:.1f} ' \
                   'seconds remaining at {rate_fmt}{postfix})'
    if image_proc_funcs:
        print('Applying custom image processing functions')
        for i, array_i in tqdm(enumerate(array),
                               total=len(ds.time),
                               leave=False,
                               bar_format=bar_format,
                               unit=' frames'):
            for func in image_proc_funcs:
                array_i = func(array_i)
            array[i, ...] = array_i

    # Clip to percentiles and rescale between 0.0 and 1.0 for plotting
    vmin, vmax = np.quantile(array[np.isfinite(array)], q=percentile_stretch)

    # Replace with vmin and vmax if present in `imshow_defaults`
    if 'vmin' in imshow_defaults:
        vmin = imshow_defaults.pop('vmin')
    if 'vmax' in imshow_defaults:
        vmax = imshow_defaults.pop('vmax')

    # Rescale between 0 and 1
    array = rescale_intensity(array,
                              in_range=(vmin, vmax),
                              out_range=(0.0, 1.0))
    array = np.squeeze(array)  # remove final axis if only one band

    # Set up figure
    fig, ax = plt.subplots()
    fig.set_size_inches(width * scale / 72, height * scale / 72, forward=True)
    fig.subplots_adjust(left=0, bottom=0, right=1, top=1, wspace=0, hspace=0)

    # Optionally add colorbar
    if show_colorbar & (len(bands) == 1):
        _add_colorbar(fig, ax, vmin, vmax, imshow_defaults, colorbar_defaults)

    # Animate
    print(f'Exporting animation to {output_path}')
    anim = FuncAnimation(
        fig=fig,
        func=_update_frames,
        fargs=(
            ax,  # axis to plot into
            [left, right, bottom, top],  # imshow extent
            annotation_list,  # list of text annotations
            show_gdf,  # geodataframe to plot over imagery
            gdf_defaults,  # any kwargs used to plot gdf
            annotation_defaults,  # kwargs for annotations
            imshow_defaults),  # kwargs for imshow
        frames=len(ds.time),
        interval=interval,
        repeat=False)



    # Export animation to file
    if Path(output_path).suffix == '.gif':
        anim.save(output_path, writer='pillow')
    else:
        anim.save(output_path, dpi=72)



In [45]:
def generate_single_masked_animations(ds, geodataframe_path, roi_name, interval, class_colour, prefix=None, list_extra_labels=[]):
    
    gdf = gpd.read_file(geodataframe_path)
    gdf['colors']='red'
    
    final_animations_dir = os.path.join(output_dir, 'final_animations')
    if not os.path.exists(final_animations_dir):
        os.makedirs(final_animations_dir)

    xr_animation_modified(ds=ds,
                bands=['nbart_red', 'nbart_green','nbart_blue'],
                output_path=f'{output_dir}/{roi_name}_timeseries_{prefix}_masked.gif',
                show_gdf=gdf,
                interval=interval,
                width_pixels=350,
                show_colorbar=False,
                show_date = '%Y',
                gdf_kwargs = {'color': class_colour},
                percentile_stretch=(0.01, 0.99),
                annotation_kwargs= {'fontsize': 25},
                list_extra_labels=list_extra_labels)
    plt.close()

    file_name_gmad = f'{output_dir}/{roi_name}_timeseries_{prefix}_masked.gif'
    
    return file_name_gmad


In [38]:
# 215: (218, 92, 105, 255, "Artificial surface")

class_colour = (218/255, 92/255, 105/255)

mask_buffer = 0.1
mask_interval = 400
mask_lat = -35.1775
mask_lon = 149.09052
mask_time = ('2000', '2022')

mask_lat_range = (mask_lat - buffer, mask_lat + buffer)
mask_lon_range = (mask_lon - buffer, mask_lon + buffer)
 
roi_name_mask = 'ACT_urban_expansion'

lvl3_geojson_fpath = 'vector_classes/canberra_urban_expansion_lc_lvl3_combined_years.geojson' #polygons of landcover classes to be animated
lvl4_geojson_fpath = 'vector_classes/canberra_urban_expansion_lc_lvl3_combined_years.geojson'

In [22]:
query ={
        'x': mask_lon_range,
        'y': mask_lat_range,
        'time': mask_time
    }

mask_gm_ds = dc.load(product=gmad_product,
                measurements=['nbart_red', 'nbart_green', 'nbart_blue'],
                **query)

In [46]:
mask_lvl3_fname = generate_single_masked_animations(mask_gm_ds, lvl3_geojson_fpath, roi_name_mask, mask_interval, class_colour, prefix='lc_lvl3',list_extra_labels = [f'{i}_TEST' for i in range(len(mask_gm_ds.time))])
mask_lvl4_fname = generate_single_masked_animations(mask_gm_ds, lvl4_geojson_fpath, roi_name_mask, mask_interval, class_colour, prefix='lc_lvl4',list_extra_labels = [f'{i}_TEST' for i in range(len(mask_gm_ds.time))])

/tmp/ipykernel_619/1997306337.py:172: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  gdf[time_col] = pd.to_datetime(gdf[time_col], errors='ignore')
/tmp/ipykernel_619/1997306337.py:172: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  gdf[time_col] = pd.to_datetime(gdf[time_col], errors='ignore')


Exporting animation to output_gifs/ACT_urban_expansion_timeseries_lc_lvl3_masked.gif


/tmp/ipykernel_619/1997306337.py:172: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  gdf[time_col] = pd.to_datetime(gdf[time_col], errors='ignore')
/tmp/ipykernel_619/1997306337.py:172: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  gdf[time_col] = pd.to_datetime(gdf[time_col], errors='ignore')


Exporting animation to output_gifs/ACT_urban_expansion_timeseries_lc_lvl4_masked.gif
